[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_17/MFM_ch17_predictable_crises/MFM_ch17_predictable_crises.ipynb)

# MFM_ch17_predictable_crises

Second case study of Chapter 17: Greenwood, Hanson, Shleifer and Sorensen (2022), Predictable Financial Crises, Journal of Finance 77(2), 863-921. On the Jorda-Schularick-Taylor Macrohistory Database (R6, 18 advanced economies, read online), forecasts made in 1950-2016: 3-year change of business and household credit to GDP, 3-year real stock and house price growth, the R-zone (credit growth in the top quintile and price growth in the top tercile of the panel, Eq. 2a-2c) and the probability that a financial crisis (JST chronology) begins within h = 1, ..., 4 years, with 95% year-block bootstrap intervals; the credit threshold is varied from the 50th to the 95th percentile.

*Modelling Financial Markets (MFM), Chapter 17: Bubbles and Crashes. Author: Daniel Traian Pele.*

The JST sample (18 economies) is smaller than the paper's 42-country panel, uses bank loans instead of total credit and the JST crisis chronology instead of Baron, Verner and Xiong (2021); the paper's own numbers are quoted only from its abstract.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_17'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import io
import json
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Course colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'
Magenta  = '#D63384'
Brown    = '#795548'

HERE = '.'
SEED = 42
R_MC = 2000
# Computing the confidence indicator takes several minutes; set RECOMPUTE = True to recompute it, otherwise precomputed values are used.
RECOMPUTE = False
QL_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_17/'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend(fig, axes, ncol=3, y=0.0):
    """One legend for a multi-panel figure, below the figure."""
    h, l = [], []
    for ax in np.atleast_1d(axes):
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l and not ll.startswith('_'):
                h.append(hh)
                l.append(ll)
    fig.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o.date())
    return o


def yrs(idx):
    """Date -> decimal years."""
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.dayofyear - 1) / 365.25)


def todate(x):
    """Ani zecimali -> data."""
    y = int(np.floor(x))
    return pd.Timestamp(f'{y}-01-01') + pd.Timedelta(days=float((x - y) * 365.25))


def d2s(d):
    return None if d is None else str(pd.Timestamp(d).date())

## Data and analysis

Data: Jorda-Schularick-Taylor Macrohistory Database, release 6 (https://www.macrohistory.net/database/), free for research and teaching with citation of Jorda, Schularick and Taylor (2017), NBER Macroeconomics Annual 31, 213-263.

In [ ]:
JST_URL = 'https://www.macrohistory.net/app/download/9834512569/JSTdatasetR6.xlsx'
T0, T1 = 1950, 2016          # anii de prognoza t (h <= 4 ani, datele JST R6 se opresc in 2020)
HMAX = 4
Q_DEBT, Q_PRICE = 0.80, 2 / 3
BLOCK, B_BOOT, BOOT_SEED = 5, 2000, 17
Q_GRID = [round(0.50 + 0.05 * i, 2) for i in range(10)]     # pragul creditului (percentila) pentru pagina interactiva
PAPER = {'rzone_3y': 0.40, 'normal_3y': 0.07}               # rezumatul articolului (JF 2022): ~40% si ~7% la 3 ani
SECTORS = {'bus': ('tbus', 'equity'), 'hh': ('thh', 'house')}

_CACHE = {}


def jst():
    """Baza JST Macrohistory R6 (online, fara cheie), ca DataFrame tara-an."""
    if 'jst' not in _CACHE:
        for attempt in range(4):
            try:
                req = urllib.request.Request(JST_URL, headers={'User-Agent': 'Mozilla/5.0'})
                raw = urllib.request.urlopen(req, timeout=180).read()
                break
            except OSError:
                if attempt == 3:
                    raise
        _CACHE['jst'] = pd.read_excel(io.BytesIO(raw), engine='openpyxl')
    return _CACHE['jst']


def panel():
    """Predictorii din ec. (2a)-(2b) si indicatorii Crisis_{t+1..t+h}, pe tari si ani."""
    d = jst().sort_values(['country', 'year']).copy()
    out = []
    for c, g in d.groupby('country'):
        g = g.set_index('year').reindex(range(g.year.min(), g.year.max() + 1))
        lcpi = np.log(g['cpi'])
        row = pd.DataFrame(index=g.index)
        row['country'] = c
        for s, (credit, _) in SECTORS.items():
            ratio = 100 * g[credit] / g['gdp']
            row[f'd_{s}'] = ratio - ratio.shift(3)
        leq = np.log1p(g['eq_capgain'])
        row['p_bus'] = 100 * (leq + leq.shift(1) + leq.shift(2) - (lcpi - lcpi.shift(3)))
        lhp = np.log(g['hpnom']) - lcpi
        row['p_hh'] = 100 * (lhp - lhp.shift(3))
        cr = g['crisisJST']
        row['onset'] = cr
        for h in range(1, HMAX + 1):
            fut = pd.concat([cr.shift(-j) for j in range(1, h + 1)], axis=1)
            row[f'c{h}'] = fut.max(axis=1).where(fut.notna().all(axis=1))
        out.append(row)
    p = pd.concat(out).rename_axis('year').reset_index()
    return p[(p.year >= T0) & (p.year <= T1)]


def sector_sample(p, s):
    cols = [f'd_{s}', f'p_{s}'] + [f'c{h}' for h in range(1, HMAX + 1)]
    return p.dropna(subset=cols)[['country', 'year'] + cols].rename(columns={f'd_{s}': 'debt', f'p_{s}': 'price'})


def flags(x, qd=Q_DEBT, qp=Q_PRICE):
    """High-Debt-Growth, High-Price-Growth, R-zone (praguri din distributia panelului, ca in articol)."""
    td, tp = float(np.quantile(x['debt'], qd)), float(np.quantile(x['price'], qp))
    hd, hp = (x['debt'] > td).values, (x['price'] > tp).values
    return hd, hp, hd & hp, td, tp


def block_index(years, rng):
    """Indicii unei replici bootstrap pe blocuri de ani calendaristici (circular)."""
    yrs = np.arange(T0, T1 + 1)
    n = len(yrs)
    starts = rng.integers(0, n, int(np.ceil(n / BLOCK)))
    pick = np.concatenate([yrs[(s + np.arange(BLOCK)) % n] for s in starts])[:n]
    by = {y: np.flatnonzero(years == y) for y in yrs}
    return np.concatenate([by[y] for y in pick])


def boot_probs(x, masks, B=B_BOOT, seed=BOOT_SEED):
    """CI 95% pentru P(Crisis_{t+1..t+h} = 1 | masca), h = 1..HMAX, pentru fiecare masca (aceleasi replici)."""
    rng = np.random.default_rng(seed)
    C = x[[f'c{h}' for h in range(1, HMAX + 1)]].values
    yrs = x['year'].values
    res = {k: np.full((B, HMAX), np.nan) for k in masks}
    for b in range(B):
        idx = block_index(yrs, rng)
        for k, m in masks.items():
            mm = m[idx]
            if mm.sum():
                res[k][b] = C[idx][mm].mean(axis=0)
    return {k: [[float(np.nanquantile(v[:, j], 0.025)), float(np.nanquantile(v[:, j], 0.975))] for j in range(HMAX)]
            for k, v in res.items()}


def boot_diff(x, m1, m0, B=B_BOOT, seed=BOOT_SEED):
    """CI 95% pentru diferenta P(. | m1) - P(. | m0), h = 1..HMAX (aceleasi replici ca boot_probs)."""
    rng = np.random.default_rng(seed)
    C = x[[f'c{h}' for h in range(1, HMAX + 1)]].values
    yrs = x['year'].values
    d = np.full((B, HMAX), np.nan)
    for b in range(B):
        idx = block_index(yrs, rng)
        a, z = m1[idx], m0[idx]
        if a.sum() and z.sum():
            d[b] = C[idx][a].mean(axis=0) - C[idx][z].mean(axis=0)
    return [[float(np.nanquantile(d[:, j], 0.025)), float(np.nanquantile(d[:, j], 0.975))] for j in range(HMAX)]


def probs(x, m):
    C = x[[f'c{h}' for h in range(1, HMAX + 1)]].values
    return [float(v) for v in C[m].mean(axis=0)]


def preceded(p, x, rz, s):
    """Crizele (anul de inceput in T0+1..T1+1) precedate de R-zone in cel putin unul din cei 3 ani anteriori."""
    on = set(map(tuple, x.loc[rz, ['country', 'year']].values))
    cr = p[(p['onset'] == 1)][['country', 'year']]
    cr = cr[(cr.year > T0) & (cr.year <= T1 + 1)]
    have = set(map(tuple, x[['country', 'year']].values))
    rows = []
    for c, y in cr.values:
        obs = [(c, y - j) in have for j in (1, 2, 3)]
        if not any(obs):
            continue                       # nicio observatie a sectorului in cei 3 ani: criza nu intra in numarator
        rows.append(dict(country=c, year=int(y), hit=any((c, y - j) in on for j in (1, 2, 3))))
    return rows


def sector_stats(p, s, qd=Q_DEBT, boot=True):
    x = sector_sample(p, s).reset_index(drop=True)
    hd, hp, rz, td, tp = flags(x, qd)
    normal = ~hd & ~hp
    allm = np.ones(len(x), bool)
    o = dict(n=int(len(x)), n_countries=int(x['country'].nunique()), first=int(x.year.min()), last=int(x.year.max()),
             thr_debt=td, thr_price=tp, n_rzone=int(rz.sum()), n_normal=int(normal.sum()),
             p_rzone=probs(x, rz), p_normal=probs(x, normal), p_all=probs(x, allm),
             )
    pre = preceded(p, x, rz, s)
    o['crises'] = pre
    o['n_crises'] = len(pre)
    o['n_preceded'] = int(sum(r['hit'] for r in pre))
    o['rzone_years'] = [dict(country=c, year=int(y)) for c, y in x.loc[rz, ['country', 'year']].values]
    if boot:
        ci = boot_probs(x, {'rzone': rz, 'normal': normal, 'all': allm})
        o['ci_rzone'], o['ci_normal'], o['ci_all'] = ci['rzone'], ci['normal'], ci['all']
        o['ci_diff'] = boot_diff(x, rz, normal)
    return o, x


def fig_ghss(res):
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), sharey=True)
    h = np.arange(1, HMAX + 1)
    names = {'bus': 'Business credit/GDP, real stock prices', 'hh': 'Household credit/GDP, real house prices'}
    for ax, s in zip(axes, ['bus', 'hh']):
        o = res[s]
        for key, col, lab, off, ls in [('rzone', IDAred, 'R-zone', -0.08, '-'),
                                       ('normal', MainBlue, 'Normal times (neither growth high)', 0.08, '-'),
                                       ('all', Forest, 'All country-years', 0.0, '--')]:
            y = 100 * np.array(o[f'p_{key}'])
            lo = 100 * np.array([c[0] for c in o[f'ci_{key}']])
            hi = 100 * np.array([c[1] for c in o[f'ci_{key}']])
            ax.errorbar(h + off, y, yerr=[y - lo, hi - y], fmt='o', ls=ls, color=col, capsize=3, ms=4, lw=1.3,
                        label=lab + ' (95% CI, year-block bootstrap)' if key == 'rzone' else lab)
        ax.plot([3.18], [100 * PAPER['rzone_3y']], 'D', mfc='none', mec=IDAred, ms=6, mew=1.2,
                label='Greenwood et al. (2022), 42 countries: R-zone, h = 3')
        ax.plot([3.18], [100 * PAPER['normal_3y']], 'D', mfc='none', mec=MainBlue, ms=6, mew=1.2,
                label='Greenwood et al. (2022): normal times, h = 3')
        ax.set_xticks(h)
        ax.set_xlabel('Horizon h (years after t)')
        ax.set_title(names[s] + f'\n{o["n_countries"]} economies, {o["n_rzone"]} R-zone years of {o["n"]}', fontsize=8.5, color='black')
    axes[0].set_ylabel('P(crisis begins in t+1 ... t+h), %')
    fig.suptitle(f'Crisis probability after credit and asset price booms, JST Macrohistory data, forecasts made in {T0}-{T1}',
                 fontsize=8.5, x=0.02, ha='left')
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    hs, ls = axes[0].get_legend_handles_labels()
    order = [ls.index(l) for l in ls if l.startswith('R-zone')] + [ls.index(l) for l in ls if l.startswith('Normal')] + \
        [ls.index(l) for l in ls if l.startswith('All')] + [ls.index(l) for l in ls if l.startswith('Greenwood')]
    fig.legend([hs[i] for i in order], [ls[i] for i in order], loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=2,
               frameon=False)
    save_fig('ch17_ghss_rzone')


def grid(p):
    """Pentru pagina interactiva: pragul creditului (percentila) variaza, pragul pretului ramane la 66,7."""
    out = {}
    for s in SECTORS:
        rows = []
        for q in Q_GRID:
            o, x = sector_stats(p, s, qd=q, boot=False)
            hd, hp, rz, td, tp = flags(x, q)
            ci = boot_probs(x, {'rzone': rz, 'normal': ~hd & ~hp})
            rows.append(dict(q=q, thr_debt=td, n_rzone=o['n_rzone'], p_rzone=o['p_rzone'], ci_rzone=ci['rzone'],
                             n_normal=o['n_normal'], p_normal=o['p_normal'], ci_normal=ci['normal'],
                             n_crises=o['n_crises'], n_preceded=o['n_preceded']))
        out[s] = rows
    return out


def main(with_grid=False):
    p = panel()
    res = {}
    for s in SECTORS:
        res[s], _ = sector_stats(p, s)
    on = {s: set((r['country'], r['year']) for r in res[s]['rzone_years']) for s in SECTORS}
    both = {}
    for s in SECTORS:
        for r in res[s]['crises']:
            both[(r['country'], r['year'])] = any((r['country'], r['year'] - j) in on[k] for k in SECTORS for j in (1, 2, 3))
    res['either'] = dict(n_crises=len(both), n_preceded=int(sum(both.values())))
    res['paper'] = PAPER
    res['sample'] = dict(t0=T0, t1=T1, block=BLOCK, B=B_BOOT, countries=int(p['country'].nunique()),
                         onsets=int(((p['onset'] == 1) & (p.year > T0) & (p.year <= T1 + 1)).sum()))
    fig_ghss(res)
    if with_grid:
        res['grid'] = grid(p)
    with open(os.path.join(OUT_DIR, 'case17_crises.json'), 'w') as f:
        json.dump(jsonable(res), f, indent=1)
    return res, p


## Run

In [ ]:
r, _ = main(with_grid=True)
for s in SECTORS:
    o = r[s]
    print(s, {k: o[k] for k in ['n', 'n_countries', 'first', 'last', 'thr_debt', 'thr_price', 'n_rzone', 'n_normal',
                                'n_crises', 'n_preceded']})
    for k in ['rzone', 'normal', 'all']:
        print('  ', k, [round(100 * v, 1) for v in o[f'p_{k}']], [[round(100 * a, 1), round(100 * b, 1)] for a, b in o[f'ci_{k}']])
    print('  diff', [[round(100 * a, 1), round(100 * b, 1)] for a, b in o['ci_diff']])
    print('  crises', [(c['country'], c['year'], c['hit']) for c in o['crises']])
print('either', r['either'], r['sample'])
for s in SECTORS:
    print(s, [(g['q'], g['n_rzone'], round(100 * g['p_rzone'][2], 1), g['n_preceded'], g['n_crises']) for g in r['grid'][s]])


![ch17_ghss_rzone](./ch17_ghss_rzone.png)

Output: `ch17_ghss_rzone.pdf`, `case17_crises.json`